In [1]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import tkinter as tk
from tkinter import ttk
import threading
import time

class MinimalPlayer:
    def __init__(self, file_path):
        self.file_path = file_path
        self.playing = False
        self.paused = False
        self.audio_data = None
        self.sample_rate = 0
        self.position = 0
        self.stream = None
        
        # Try to load the file
        try:
            self.audio_data, self.sample_rate = sf.read(self.file_path, dtype='float32')
            print(f"Loaded file: {self.file_path}")
        except Exception as e:
            print(f"Could not load file: {e}")
    
    def play(self):
        if self.playing and self.paused:
            # Resume
            self.paused = False
            return True
            
        if self.playing:
            return False  # Already playing
            
        if self.audio_data is None:
            return False  # No audio loaded
        
        # Create playback thread
        self.playing = True
        self.paused = False
        threading.Thread(target=self._play_thread, daemon=True).start()
        return True
        
    def _play_thread(self):
        try:
            # Find the Output 1/2 device or default
            device_id = None
            devices = sd.query_devices()
            
            # Look for Output 1/2
            for i, dev in enumerate(devices):
                if dev['max_output_channels'] > 0 and 'output 1/2' in dev['name'].lower():
                    device_id = i
                    break
            
            # If not found, use default
            if device_id is None:
                device_id = sd.default.device[1]
                
            # Play through the selected device
            self.position = 0
            
            # Simple blocking playback
            sd.play(self.audio_data, self.sample_rate, device=device_id, blocking=False)
            
            # Wait until finished
            while sd.get_stream().active and self.playing and not self.paused:
                time.sleep(0.1)
                
            if self.paused:
                sd.stop()
            else:
                self.playing = False
                
        except Exception as e:
            print(f"Playback error: {e}")
            self.playing = False
            self.paused = False
    
    def pause(self):
        if self.playing and not self.paused:
            self.paused = True
            return True
        return False
    
    def stop(self):
        if self.playing:
            self.playing = False
            self.paused = False
            sd.stop()
            return True
        return False
        
    def restart(self):
        self.stop()
        time.sleep(0.1)
        return self.play()
        
    def cleanup(self):
        self.stop()
        sd.stop()


class MinimalPlayerApp:
    def __init__(self, root):
        self.root = root
        root.title("Instructions Player")
        root.geometry("300x150")
        
        # Create a player
        self.player = MinimalPlayer(r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\GeneralInstructions.wav")
        
        # Close handler
        root.protocol("WM_DELETE_WINDOW", self.on_close)
        
        # Create widgets
        main_frame = ttk.Frame(root, padding="10")
        main_frame.pack(expand=True, fill=tk.BOTH)
        
        # Title
        ttk.Label(main_frame, text="Instructions Player", font=("Arial", 12, "bold")).pack(pady=5)
        
        # Buttons
        button_frame = ttk.Frame(main_frame)
        button_frame.pack(pady=10)
        
        self.play_btn = ttk.Button(button_frame, text="Play Instructions", command=self.play, width=16)
        self.play_btn.grid(row=0, column=0, padx=5, pady=5)
        
        self.pause_btn = ttk.Button(button_frame, text="Pause", command=self.pause, width=8)
        self.pause_btn.grid(row=0, column=1, padx=5, pady=5)
        
        self.restart_btn = ttk.Button(button_frame, text="Restart", command=self.restart, width=8)
        self.restart_btn.grid(row=0, column=2, padx=5, pady=5)
        
        # Status
        self.status_var = tk.StringVar(value="Ready")
        ttk.Label(main_frame, textvariable=self.status_var).pack(pady=5)
    
    def play(self):
        if self.player.play():
            if self.player.paused:
                self.status_var.set("Playing")
                self.pause_btn.config(text="Pause")
            else:
                self.status_var.set("Playing")
        else:
            self.status_var.set("Already playing")
    
    def pause(self):
        if self.player.playing and not self.player.paused:
            if self.player.pause():
                self.status_var.set("Paused")
                self.pause_btn.config(text="Resume")
        else:
            self.play()  # Resume
    
    def restart(self):
        if self.player.restart():
            self.status_var.set("Playing from start")
            self.pause_btn.config(text="Pause")
    
    def on_close(self):
        self.player.cleanup()
        self.root.destroy()


# Run the application
if __name__ == "__main__":
    root = tk.Tk()
    app = MinimalPlayerApp(root)
    root.mainloop()

Loaded file: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\GeneralInstructions.wav
